#Build RAG Using LlamaIndex

### Load all the required libraries

In [11]:
!pip install -U -q llama-index llama-index-vector-stores-chroma llama-index-llms-huggingface-api llama-index-embeddings-huggingface pypdf llama-index-readers-file

### Set OpenAI API Key

In [12]:
import os
os.environ["OPENAI_API_KEY"] = ""

### Login to HuggingFace

In [13]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN') # Access the secret
if HF_TOKEN:
    login(HF_TOKEN)
    print("Successfully logged in to Hugging Face!")
else:
    print("Token not found or notebook access not enabled.")

Successfully logged in to Hugging Face!


### Build Simple RAG system

In [14]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

# Load document
reader = SimpleDirectoryReader(input_files=["/content/the-state-of-ai-how-organizations-are-rewiring-to-capture-value_final.pdf"])
documents = reader.load_data()
print(f"Loaded {len(documents)} document(s).")

# Split into chunks
splitter = SentenceSplitter(chunk_size=1024)
nodes = splitter.get_nodes_from_documents(documents)

# Set up LLM and embedding model
Settings.llm = OpenAI(model="gpt-3.5-turbo")
Settings.embed_model = OpenAIEmbedding(model="text-embedding-ada-002")

# Create vector index
vector_index = VectorStoreIndex(nodes)

# Create query engine
query_engine = vector_index.as_query_engine()

Loaded 26 document(s).


###Inspecting the vector store

In [15]:
# Access the vector store data directly
vector_store = vector_index.vector_store

# Get embedding dictionary and node dictionary
embedding_dict = vector_store.data.embedding_dict
node_dict = vector_store.data.text_id_to_ref_doc_id

print(f"Number of embeddings: {len(embedding_dict)}")
print(f"Number of node references: {len(node_dict)}")

# Show first few embeddings
for i, (node_id, embedding) in enumerate(list(embedding_dict.items())[:3]):
    print(f"\n--- Embedding {i} ---")
    print(f"Node ID: {node_id}")
    print(f"Embedding dimension: {len(embedding)}")
    print(f"First 10 values: {embedding[:10]}")

Number of embeddings: 27
Number of node references: 27

--- Embedding 0 ---
Node ID: bfabc68c-a304-4140-852d-fdcac9cd7407
Embedding dimension: 1536
First 10 values: [-0.013638702221214771, -0.008868308737874031, -0.009905046783387661, -0.018465135246515274, 0.0038247208576649427, 0.009470737539231777, -0.029589049518108368, 0.0037546709645539522, -0.0331195592880249, -0.014108036644756794]

--- Embedding 1 ---
Node ID: 71ed07b8-72c4-4943-924e-5a0b5e3735be
Embedding dimension: 1536
First 10 values: [-0.015782957896590233, -0.016759799793362617, -0.002569440519437194, -0.017387768253684044, -0.0031433345284312963, 0.016955167055130005, -0.028272567316889763, 0.005519151221960783, -0.022327791899442673, -0.02260688878595829]

--- Embedding 2 ---
Node ID: c95e33f9-76a1-48ee-b01b-0b0cc524764e
Embedding dimension: 1536
First 10 values: [0.0022242495324462652, -0.026832429692149162, -0.00385244726203382, -0.03319031372666359, -0.008896998129785061, 0.018022987991571426, -0.014857515692710876,

###Asking questions to the RAG system

In [16]:
# Query the document
response = query_engine.query("Who is Lareina Yee?")
print(response)

Lareina Yee is a Senior partner and McKinsey Global Institute director.


###Checking if the responses make sense

In [17]:
print(len(response.source_nodes))

2


In [18]:
# Print out each source node
print("Source nodes:")
print("=" * 50)

for i, node in enumerate(response.source_nodes):
    print(f"Node {i+1}:")
    print(f"Score: {node.score}")
    print(f"Text: {node.text}")
    print(f"Metadata: {node.metadata}")
    print("-" * 30)

Source nodes:
Node 1:
Score: 0.7095338963009279
Text: The state of AI  
March 2025
Alex Singla  
Alexander Sukharevsky  
Lareina Yee  
Michael Chui  
Bryce Hall
How organizations are rewiring to capture value
Metadata: {'page_label': '1', 'file_name': 'the-state-of-ai-how-organizations-are-rewiring-to-capture-value_final.pdf', 'file_path': '/content/the-state-of-ai-how-organizations-are-rewiring-to-capture-value_final.pdf', 'file_type': 'application/pdf', 'file_size': 5564174, 'creation_date': '2026-07-02', 'last_modified_date': '2026-07-02'}
------------------------------
Node 2:
Score: 0.7052493750805306
Text: McKinsey commentary
Lareina Yee
Senior partner and McKinsey Global Institute director
Although we remain in the early stages of gen AI, we’re beginning to get a glimpse into 
the ways the technology is affecting the workforce. A common fear about the technology 
is that it will be a job killer, as organizations offload tasks historically done by employees to 
increasingly power

In [19]:
# Ask more questions
response2 = query_engine.query("What are the main findings about AI adoption?")
print(response2)

The main findings about AI adoption are that the reported use of AI has been increasing, with 78 percent of respondents stating their organizations use AI in at least one business function. The use of AI is most commonly reported in IT and marketing and sales functions. Additionally, organizations are using AI in more business functions than before, with most survey respondents reporting the use of AI in more than one business function on average. Despite the increase in AI adoption, less than one-third of respondents report that their organizations are following most of the adoption and scaling practices for gen AI.


In [20]:
response3 = query_engine.query("What does the document say about AI risks?")
print(response3)

The document highlights that organizations are increasingly working to mitigate risks associated with generative AI. Specifically, they are focusing on addressing risks related to inaccuracy, cybersecurity, and intellectual property infringement, which are among the most commonly reported risks causing negative consequences for organizations. Additionally, larger organizations are reported to be more actively managing potential cybersecurity and privacy risks compared to other organizations.
